In [1]:
from pathlib import Path

arquivo = Path("../data/raw/bea/personal_income.csv")

with open(arquivo, "r", encoding="utf-8-sig") as f:
    for numero, linha in enumerate(f):
        if numero >= 15:
            break
        print(f"{numero:02d}: {linha.rstrip()}")

00: "SAINC1 State annual personal income summary: personal income, population, per capita personal income"
01: Per capita personal income 2 (Dollars)
02: State or DC
03: GeoFIPS,GeoName,1929,1930,1931,1932,1933,1934,1935,1936,1937,1938,1939,1940,1941,1942,1943,1944,1945,1946,1947,1948,1949,1950,1951,1952,1953,1954,1955,1956,1957,1958,1959,1960,1961,1962,1963,1964,1965,1966,1967,1968,1969,1970,1971,1972,1973,1974,1975,1976,1977,1978,1979,1980,1981,1982,1983,1984,1985,1986,1987,1988,1989,1990,1991,1992,1993,1994,1995,1996,1997,1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
04: 00000,United States,699,621,528,402,375,427,477,539,579,531,561,600,730,930,1133,1218,1261,1278,1340,1448,1406,1533,1704,1793,1869,1855,1952,2059,2138,2160,2260,2321,2383,2498,2589,2735,2918,3134,3328,3617,3931,4198,4471,4857,5363,5836,6324,6875,7516,8356,9232,10184,11291,11990,12701,13889,14746,15482,16228,17442,18658,1961

In [2]:
import pandas as pd

bea = pd.read_csv(
    arquivo,
    skiprows=3,
    encoding="utf-8-sig",
    dtype={"GeoFIPS": str},
    low_memory=False
)

anos = [str(ano) for ano in range(2019, 2025)]

colunas = ["GeoFIPS", "GeoName"] + anos

bea = bea[colunas].copy()

print("Dimensões:", bea.shape)
display(bea.head())

Dimensões: (66, 8)


,GeoFIPS,GeoName,2019,2020,2021,2022,2023,2024
0,00000,United States,55567.0,59151.0,64718.0,66316.0,70191.0,73634.0
1,01000,Alabama,42996.0,45874.0,50651.0,51706.0,54760.0,57167.0
2,02000,Alaska *,60483.0,61892.0,65773.0,69151.0,72563.0,76842.0
3,04000,Arizona,47646.0,51996.0,57116.0,59055.0,63159.0,65563.0
4,05000,Arkansas,43745.0,47140.0,53374.0,53718.0,56569.0,59403.0


In [3]:
bea_long = bea.melt(
    id_vars=["GeoFIPS", "GeoName"],
    value_vars=anos,
    var_name="year",
    value_name="income_per_capita"
)

bea_long = bea_long.rename(
    columns={"GeoName": "state"}
)

bea_long["state"] = (
    bea_long["state"]
    .str.replace(r"\s*\*$", "", regex=True)
    .str.strip()
)

bea_long["year"] = bea_long["year"].astype(int)

display(bea_long.head(10))
print("Dimensões:", bea_long.shape)

,GeoFIPS,state,year,income_per_capita
0,00000,United States,2019,55567.0
1,01000,Alabama,2019,42996.0
2,02000,Alaska,2019,60483.0
3,04000,Arizona,2019,47646.0
4,05000,Arkansas,2019,43745.0
5,06000,California,2019,64230.0
6,08000,Colorado,2019,61242.0
7,09000,Connecticut,2019,74167.0
8,10000,Delaware,2019,53776.0
9,11000,District of Columbia,2019,83429.0


Dimensões: (396, 4)


In [4]:
# Recuperar os 50 estados utilizados no CDC
cdc = pd.read_csv(
    "../data/raw/cdc_homicides_2019_2024.csv"
)

estados_validos = set(cdc["name"]) - {"District of Columbia"}

# Filtrar as unidades geográficas
bea_filtrado = bea_long[
    bea_long["state"].isin(estados_validos)
].copy()

print("Dimensões:", bea_filtrado.shape)
print("Estados:", bea_filtrado["state"].nunique())

display(
    bea_filtrado.groupby("year")["state"].nunique()
)

Dimensões: (300, 4)
Estados: 50


year
2019    50
2020    50
2021    50
2022    50
2023    50
2024    50
Name: state, dtype: int64

In [5]:
# Garantir que a renda esteja em formato numérico
bea_filtrado["income_per_capita"] = pd.to_numeric(
    bea_filtrado["income_per_capita"],
    errors="coerce"
)

print("Duplicações:",
      bea_filtrado.duplicated(["state", "year"]).sum())

print("Rendas ausentes:",
      bea_filtrado["income_per_capita"].isna().sum())

print("Rendas inválidas:",
      (bea_filtrado["income_per_capita"] <= 0).sum())

print("\nEstatísticas descritivas:")
display(bea_filtrado["income_per_capita"].describe())

Duplicações: 0
Rendas ausentes: 0
Rendas inválidas: 0

Estatísticas descritivas:


count      300.000000
mean     62559.503333
std      10626.952624
min      39157.000000
25%      54818.500000
50%      61520.500000
75%      68894.500000
max      97810.000000
Name: income_per_capita, dtype: float64

In [6]:
from pathlib import Path

bea_final = bea_filtrado[
    ["state", "year", "income_per_capita"]
].copy()

destino = Path("../data/processed/bea_income_2019_2024.csv")

bea_final.to_csv(destino, index=False)

print("Dimensões:", bea_final.shape)
print("Salvo em:", destino.resolve())

Dimensões: (300, 3)
Salvo em: /media/jandersongustavo/f8223ea0-bc74-41e8-8f66-650dbe31ee07/Arquivos_Janderson/Documentos/Projetos/us-gun-policy-ml/data/processed/bea_income_2019_2024.csv


NameError: name 'df_v3' is not defined